# AutoAugment — Learning Augmentation Strategies from Data (2018)

_Papers · Computer Vision_

**Treat data augmentation as a search problem: let reinforcement learning discover which image transforms (and how strong, how often) raise validation accuracy.**

---

This notebook is a practice scaffold for the **AutoAugment — Learning Augmentation Strategies from Data (2018)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the CIFAR10 dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
print("dataset: CIFAR10   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import random
import torch
import torch.nn as nn
import torchvision, torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image

torch.manual_seed(0); random.seed(0)

# --- 0. Recompute the worked example: search-space size + one sub-policy apply. ---
K, n_mag, n_prob = 16, 10, 11                 # 16 transforms, 10 magnitudes, 11 probabilities (Sec 3)
ops_per_subpolicy, n_subpolicies = 2, 5
exponent = ops_per_subpolicy * n_subpolicies  # = 10
n_policies = (K * n_mag * n_prob) ** exponent
print("one operation choices :", K * n_mag * n_prob)              # 1760
print("one sub-policy choices :", (K * n_mag * n_prob) ** 2)      # ~3.10e6
print("search-space size      : (16*10*11)**10 =", f"{n_policies:.3e}")  # ~2.9e32

# Walk one sub-policy [(Rotate,p=0.8), (Solarize,p=0.3)] with fixed draws from the lesson.
def apply_op(img, name, p, m, r):             # r = the drawn uniform in [0,1)
    if r >= p:
        return img, "skip"                    # operation fires only if r < p
    if name == "Rotate":   return TF.rotate(img, 30.0 * m / 9.0), "apply"   # m=9 -> ~30 deg
    if name == "Solarize": return TF.solarize(img, 256 - int(256 * m / 9)), "apply"
    return img, "apply"

dummy = Image.new("RGB", (32, 32), (120, 120, 120))
_, d1 = apply_op(dummy, "Rotate",   0.8, 9, r=0.42)   # 0.42 < 0.8 -> apply
_, d2 = apply_op(dummy, "Solarize", 0.3, 5, r=0.55)   # 0.55 >= 0.3 -> skip
print("sub-policy walk: Rotate ->", d1, "| Solarize ->", d2)   # apply | skip


# --- 1. AutoAugment policy machinery, built by hand (Sec 3 search space). ---
# Each operation: (transform_name, probability p, magnitude m in 0..9).
# A sub-policy is 2 operations; a policy is 5 sub-policies. We hand-fix a policy
# (we are NOT running the RL search that would discover one).
POLICY = [
    [("Rotate",   0.8, 9), ("Solarize",   0.3, 5)],
    [("ShearX",   0.6, 4), ("Color",      1.0, 8)],
    [("TranslateY", 0.5, 6), ("Sharpness", 0.7, 7)],
    [("Equalize", 0.4, 0), ("Contrast",   0.6, 5)],
    [("Posterize", 0.7, 4), ("Brightness", 0.5, 6)],
]

def run_op(img, name, p, m):
    if random.random() >= p:                  # fire with probability p (Sec 3)
        return img
    lo_hi = {"Color":(0.1,1.9),"Contrast":(0.1,1.9),"Brightness":(0.1,1.9),"Sharpness":(0.1,1.9)}
    if name == "Rotate":     return TF.rotate(img, 30.0 * m / 9.0)
    if name == "ShearX":     return TF.affine(img, angle=0, translate=[0,0], scale=1.0, shear=[17.0*m/9.0,0.0])
    if name == "TranslateY": return TF.affine(img, angle=0, translate=[0, int(10*m/9.0)], scale=1.0, shear=[0,0])
    if name == "Solarize":   return TF.solarize(img, 256 - int(256*m/9))
    if name == "Posterize":  return TF.posterize(img, max(1, 8 - int(4*m/9)))
    if name == "Equalize":   return TF.equalize(img)
    if name in lo_hi:
        lo, hi = lo_hi[name]; f = lo + (hi-lo)*m/9.0
        return {"Color":TF.adjust_saturation,"Contrast":TF.adjust_contrast,
                "Brightness":TF.adjust_brightness,"Sharpness":TF.adjust_sharpness}[name](img, f)
    return img

class AutoAugmentPolicy:
    "Apply ONE randomly chosen sub-policy per image (Sec 3)."
    def __init__(self, policy): self.policy = policy
    def __call__(self, img):
        sub = random.choice(self.policy)      # pick 1 of 5 sub-policies, uniform
        for name, p, m in sub:                # apply its (<=2) ops in order
            img = run_op(img, name, p, m)
        return img


# --- 2. A tiny CNN. ---
class TinyCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1,bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32,64,3,stride=2,padding=1,bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.head = nn.Linear(64, n_classes)
    def forward(self, x):
        return self.head(self.net(x).mean(dim=(2,3)))   # global average pool + linear head


# --- 3. CIFAR-10 subset; train pipeline toggles the policy for the ablation. ---
NORM = T.Normalize((0.4914,0.4822,0.4465),(0.247,0.243,0.261))
def make_loaders(use_policy):
    train_tfms = [AutoAugmentPolicy(POLICY)] if use_policy else []
    train_tfm = T.Compose(train_tfms + [T.ToTensor(), NORM])   # policy BEFORE ToTensor
    val_tfm   = T.Compose([T.ToTensor(), NORM])                # val gets NO augmentation
    tr = torch.utils.data.Subset(
        torchvision.datasets.CIFAR10("./data", True,  download=True, transform=train_tfm), range(5000))
    te = torch.utils.data.Subset(
        torchvision.datasets.CIFAR10("./data", False, download=True, transform=val_tfm),   range(2000))
    return (torch.utils.data.DataLoader(tr, batch_size=128, shuffle=True),
            torch.utils.data.DataLoader(te, batch_size=256))

device = "cuda" if torch.cuda.is_available() else "cpu"

def run(use_policy, epochs=8):
    torch.manual_seed(0)
    trl, tel = make_loaders(use_policy)
    net = TinyCNN().to(device)
    opt = torch.optim.SGD(net.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
    lf  = nn.CrossEntropyLoss()
    for _ in range(epochs):
        net.train()
        for xb, yb in trl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); lf(net(xb), yb).backward(); opt.step()
    net.eval(); correct = tot = 0
    with torch.no_grad():
        for xb, yb in tel:
            xb, yb = xb.to(device), yb.to(device)
            correct += (net(xb).argmax(1) == yb).sum().item(); tot += yb.size(0)
    return correct / tot

acc_aug = run(use_policy=True)
acc_raw = run(use_policy=False)             # ABLATION: same net, no augmentation
print(f"with AutoAugment policy : val acc {acc_aug:.3f}")
print(f"no augmentation (ablate): val acc {acc_raw:.3f}")
print("Augmentation adds variety, not labels -> usually higher val acc on a small set.")
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does applying a fixed AutoAugment-style policy raise a tiny classifier's validation accuracy versus no augmentation, and how large is the search space the RL controller must cover?_

In [ ]:
import random, math
import torch, torch.nn as nn, torchvision, torchvision.transforms as T
import torchvision.transforms.functional as TF

torch.manual_seed(0); random.seed(0)

# (Bottom chart) search-space size vs number of sub-policies: log10 of #policies.
base = 16 * 10 * 11                         # 1760 choices per operation
for s in range(1, 6):
    print(f"{s} sub-policies -> log10(#policies) = {2*s*math.log10(base):.2f}")
# 5 -> 32.45, i.e. (16*10*11)**10 ~= 2.9e32 (Sec 3)

# (Top chart) ablation: fixed AutoAugment-style policy vs no augmentation.
POLICY = [
    [("Rotate",0.8,9),("Solarize",0.3,5)], [("ShearX",0.6,4),("Color",1.0,8)],
    [("TranslateY",0.5,6),("Sharpness",0.7,7)], [("Equalize",0.4,0),("Contrast",0.6,5)],
    [("Posterize",0.7,4),("Brightness",0.5,6)],
]
def run_op(img, name, p, m):
    if random.random() >= p: return img
    f = {"Color":TF.adjust_saturation,"Contrast":TF.adjust_contrast,
         "Brightness":TF.adjust_brightness,"Sharpness":TF.adjust_sharpness}
    if name=="Rotate":     return TF.rotate(img, 30.0*m/9.0)
    if name=="ShearX":     return TF.affine(img,0,[0,0],1.0,[17.0*m/9.0,0.0])
    if name=="TranslateY": return TF.affine(img,0,[0,int(10*m/9.0)],1.0,[0,0])
    if name=="Solarize":   return TF.solarize(img, 256-int(256*m/9))
    if name=="Posterize":  return TF.posterize(img, max(1,8-int(4*m/9)))
    if name=="Equalize":   return TF.equalize(img)
    if name in f:          return f[name](img, 0.1+1.8*m/9.0)
    return img
class AA:
    def __call__(self, img):
        for name,p,m in random.choice(POLICY): img = run_op(img,name,p,m)
        return img

NORM = T.Normalize((0.4914,0.4822,0.4465),(0.247,0.243,0.261))
def loaders(use):
    tr_tfm = T.Compose(([AA()] if use else []) + [T.ToTensor(), NORM])
    va_tfm = T.Compose([T.ToTensor(), NORM])
    tr = torch.utils.data.Subset(torchvision.datasets.CIFAR10("./data",True,download=True,transform=tr_tfm), range(5000))
    va = torch.utils.data.Subset(torchvision.datasets.CIFAR10("./data",False,download=True,transform=va_tfm), range(2000))
    return torch.utils.data.DataLoader(tr,128,shuffle=True), torch.utils.data.DataLoader(va,256)
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(3,32,3,padding=1,bias=False),nn.BatchNorm2d(32),nn.ReLU(),
                                 nn.Conv2d(32,64,3,stride=2,padding=1,bias=False),nn.BatchNorm2d(64),nn.ReLU())
        self.head = nn.Linear(64,10)
    def forward(self,x): return self.head(self.net(x).mean(dim=(2,3)))
def run(use):
    torch.manual_seed(0); trl,val = loaders(use); net = TinyCNN()
    opt = torch.optim.SGD(net.parameters(),lr=0.05,momentum=0.9,weight_decay=5e-4); lf = nn.CrossEntropyLoss()
    for _ in range(8):
        net.train()
        for xb,yb in trl: opt.zero_grad(); lf(net(xb),yb).backward(); opt.step()
    net.eval(); c=t=0
    with torch.no_grad():
        for xb,yb in val: c += (net(xb).argmax(1)==yb).sum().item(); t += yb.size(0)
    return c/t
print("no aug             : acc", round(run(False),3))
print("AutoAugment policy : acc", round(run(True),3))
# Policy -> higher val acc on the small subset; numbers are ours, not the paper's.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a tiny CNN that trains on a small image subset with a fixed
            AutoAugment-style policy. Remove the policy (train on raw images, everything else identical) and
            retrain. What do you expect to happen to validation accuracy, and what does the comparison
            demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Remove only the policy transform from the training pipeline; keep the same model, depth, optimizer, epochs, and data, and keep the val pipeline unchanged. — _An honest ablation changes exactly one thing &mdash; the augmentation &mdash; so any accuracy difference is attributable to it._
- Retrain and compare final validation accuracy: with-policy vs no-aug. — _On a small dataset the model overfits quickly; augmentation injects variety that should narrow the train/val gap._
- Note augmentation adds variety, not new labels &mdash; it makes the same images look different across epochs. — _That is the mechanism by which it reduces overfitting and (usually) raises val accuracy on small data._

**Answer:** The with-policy run should reach equal-or-higher validation accuracy than the
                 no-aug run, especially on a small dataset where overfitting is severe. Because the two runs
                 differ only by the augmentation, the comparison isolates the policy as the cause &mdash;
                 the same qualitative effect the paper reports (augmentation lowers error), reproduced on a toy
                 scale. The CODEVIZ panel shows this with/without contrast on our small run.

</details>

**Problem 2.** Search-space combinatorics. Suppose a simplified AutoAugment variant used only $K=8$
            transforms, $5$ magnitude levels, $6$ probability levels, sub-policies of $2$ operations, and a
            policy of $3$ sub-policies. How many distinct policies are there? Give the formula and an
            order-of-magnitude.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count one operation: $8\times5\times6 = 240$ choices. — _Each operation independently picks a transform, a magnitude, and a probability &mdash; multiply the counts._
- Operations per policy $= 2\times3 = 6$ (2 per sub-policy, 3 sub-policies). — _Each operation is chosen independently, so the policy count is (one-operation count) raised to the number of operations._
- Policies $= 240^{6}$. Compute: $\log_{10}240\approx2.380$, so $6\times2.380 = 14.28$, giving $\approx1.9\times10^{14}$. — _Same product-to-a-power structure as the paper's $(16\times10\times11)^{10}$._

**Answer:** $(8\times5\times6)^{2\times3} = 240^{6}\approx1.9\times10^{14}$ distinct policies. The
                 structure mirrors the paper: count one operation's independent choices, then raise to the total
                 number of operations (ops-per-sub-policy $\times$ sub-policies). Even this shrunken space is
                 too large to grid-search &mdash; the very reason AutoAugment uses an RL controller.

</details>

**Problem 3.** Applying a sub-policy. A sampled sub-policy is
            [(ShearX, p=0.6, m=4), (Color, p=1.0, m=8)]. For a given image you draw uniform
            randoms $r_1=0.73$ for the first operation and $r_2=0.05$ for the second. Which operations are
            applied, and what is the resulting image?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Operation 1 (ShearX, $p=0.6$): compare $r_1=0.73$ to $0.6$. Since $0.73\ge0.6$, skip ShearX. — _An operation fires only when the drawn random is below its probability $p$._
- Operation 2 (Color, $p=1.0$): compare $r_2=0.05$ to $1.0$. Since $0.05\lt1.0$, apply Color at magnitude $8$. — _A probability of $1.0$ means the operation always fires regardless of the draw._
- Compose in order: ShearX skipped, then Color applied &rarr; the image is the original with a strong color/saturation shift, no shear. — _Sub-policy operations are applied sequentially to the (possibly already transformed) image._

**Answer:** ShearX is skipped ($0.73\ge0.6$); Color is applied at magnitude $8$
                 ($0.05\lt1.0$, and $p=1.0$ always fires). The output is the original image with a strong color
                 shift and no shear. On a different epoch the draws differ, so the same image yields a different
                 augmented view &mdash; the per-epoch variety AutoAugment relies on.

</details>